# Go Fish No Fish

### Binary classification of images (fish / no fish) using CNN in Keras Tensorflow

In this notebook I'm using Kaggle's Blue Bot Fish No Fish dataset. 

My goal is to build a CNN to classify the images as 'contains fish' or 'does not contain fish'. 

Kylie DeFeo  
DSC 344 - Intro to Machine Learning

### Context and Background

Kaggle Dataset
- Carribean Fish 
- Kaggle comepetition 

##### Problem:
Say a marine biologist is given a data set with images to identify and count fish, but only half the images actually contain fish and they're unlabeled. They can save time by classifying images into a "contains fish" category and avoid combing through images that don't have fish for them to identify. 

##### Goal:
Classify images as "contains fish" or "does not contain fish".


### Overview 
- Setup 
- Data Integration and Upload 
- Looking at the data 
- Intro to Preprocessing Practices 
- Standardization / Normalization 
- Model 
    - terms 
    - step by step
    - analysis
    - data augmentation 
    - dropout 
    - predicitions 
- Original model 
    - step by step 
    - analysis / what went wrong
    - "improvements"
- References

### Setup Packages

In [ ]:
# setup 

import numpy as np
import os
import tensorflow as tf

#tensorflow setup
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image

#visualization
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

### Import, Preprocess and Clean Data

#### Preprocessing 

For CNN we need to process the images to make them all the same size, color values and we may need to augment the data too. 
A lot of this is built in to keras preprocessing, but not all, I'll show you a model that incorporates some good practices first, then go over my original model that didn't do so well.

In [ ]:
#will reference these three variables later on 
img_height = 150
img_width = 150
batch_size = 32

# Data path - update this based on your environment
# For Kaggle: '/kaggle/input/doesimagehavefish'
# For local: path to your downloaded dataset
DATA_DIR = '/kaggle/input/doesimagehavefish'

In [ ]:
#train dataset 
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='int',
    color_mode='rgb', 
    batch_size=32, 
    image_size=(150,150),
    shuffle=True, 
    seed=42, 
    validation_split=0.20, 
    subset="training", 
)
                                         
test_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='int',
    color_mode='rgb',
    batch_size=32, 
    image_size=(150,150),
    shuffle=True, 
    seed=42, 
    validation_split=0.20, 
    subset="validation"   
)

### Making sure the data looks right by printing names/ images

In [ ]:
class_names = train_dataset.class_names
print(class_names)

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

### Measures to Increase Performance

- Dataset.cache keeps the images in memory after they're loaded off disk during the first epoch. This will ensure the dataset does not become a bottleneck while training your model. If your dataset is too large to fit into memory, you can also use this method to create a performant on-disk cache.
- Dataset.prefetch overlaps data preprocessing and model execution while training.

In [ ]:
#use autotune to better allocate run time per parameter/ step 

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

### Standardizing the data

So all images are the same size, RGB  values are [0, 255] range standardize values to be in the [0, 1] range so the inputs are smaller and more managable for the model.

In [ ]:
#keras.layers rescaling is built in 

normalization_layer = layers.Rescaling(1./255)

In [ ]:
normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(normalized_ds))
first_image = image_batch[0]
# Notice the pixel values are now in `[0,1]`.
print(np.min(first_image), np.max(first_image))

# Model 

### Terms

**Epochs:** one pass over entire train dataset (contains many steps)

**Steps:** batches of samples to train (not necessary with limited train data)

**Layers:** *Convolutional:* weighted sum of pixel values *MaxPool:* Reducing output size of convolutional layers (simplifying)

**Cv2:** Open source computer vision package for image displays etc

**Conv2d:** Two dimensional convolution layer used in CNN

**CNN:** Convolutional Neural Network - feed forward artificial neural network with variations of multi layer perceptrons to minimize preprocesssing


#### Notes

I will refer to standardized model as 'modelt' which is based off of an intro CNN from tensorflow.

In [ ]:
#tensorflow model from tensorflow documentation

num_classes = len(class_names)

modelt = keras.Sequential([
    layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes)
])

In [ ]:
modelt.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
modelt.summary()

#### Training the model

In [ ]:
#29ms/ step
#<2 min to run

epochs=10
historyt = modelt.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

# Analyze the model 

**adam optimizer:** optimization method within Keras CNN, works differently than stochastic gradient descent, changes the learning rate as it goes by using a combination of an adaptive gradient algorithm (AdaGrad) and Root Mean Sqaure Propogation (RMSProp), it performs well in neural networks and is widely used.

**cross entropy:** for one hot encoded data (0/1, fish/ nonfish), takes a target value *t* and predicted value *t* penalizes for false predictions and penalizes heavier for confident false predictions 

**accuracy / val accuracy:** simple accuracy of how well the model predicted, useful to compare train and validation results to catch overfitting

**loss / val loss:** sum of errors made when prediciting class

#### Visualizing model t results

In [ ]:
acc = historyt.history['accuracy']
val_acc = historyt.history['val_accuracy']

loss = historyt.history['loss']
val_loss = historyt.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

#was overfitting a lot before normailzation

### Preprocessing methods

#### Data augmentation on model t to reduce overfitting

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal",
                          input_shape=(img_height,
                                      img_width,
                                      3)),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ]
)

In [ ]:
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    for i in range(9):
        augmented_images = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_images[0].numpy().astype("uint8"))
        plt.axis("off")

#### Dropout 

Another technique to reduce overfitting is to introduce dropout regularization to the network.

When you apply dropout to a layer, it randomly drops out (by setting the activation to zero) a number of output units from the layer during the training process. Dropout takes a fractional number as its input value, in the form such as 0.1, 0.2, 0.4, etc. This means dropping out 10%, 20% or 40% of the output units randomly from the applied layer.

Let's create a new neural network with tf.keras.layers.Dropout before training it using the augmented images:

In [ ]:
modelt = keras.Sequential([
    data_augmentation,
    layers.Rescaling(1./255),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes)
])

In [ ]:
modelt.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

modelt.summary()

In [ ]:
#37 ms/ step
#3 min to run
epochs = 15
historyt = modelt.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

#### Visualize again after augmenting and dropout

In [ ]:
acc = historyt.history['accuracy']
val_acc = historyt.history['val_accuracy']

loss = historyt.history['loss']
val_loss = historyt.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

#looks better!

In [ ]:
plt.plot(historyt.history['accuracy'], label='accuracy')
plt.plot(historyt.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend(loc='lower right')

#### Predict on a new image

In [ ]:
# To test with your own image, upload it and update the path
# img_path = 'your_image.jpeg'

# Example prediction function
def predict_image(img_path):
    img = tf.keras.utils.load_img(
        img_path, target_size=(img_height, img_width)
    )
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0) # Create a batch

    predictions = modelt.predict(img_array)
    score = tf.nn.softmax(predictions[0])

    plt.imshow(img)
    print(
        "This image most likely belongs to {} with a {:.2f} percent confidence."
        .format(class_names[np.argmax(score)], 100 * np.max(score))
    )

# Original model 
## (no scaling, little preprocessing, poor results)

Below is my original model that didn't perform as well. I'm keeping it here to show the comparison and what I learned about the importance of preprocessing.

In [ ]:
model = keras.Sequential()

# convolutional layer and maxpool layer 1
model.add(keras.layers.Conv2D(32,(3,3),activation='relu',input_shape=(150,150,3)))
model.add(keras.layers.MaxPool2D(2,2))

# convolutional layer and maxpool layer 2
model.add(keras.layers.Conv2D(64,(3,3),activation='relu'))
model.add(keras.layers.MaxPool2D(2,2))

# convolutional layer and maxpool layer 3
model.add(keras.layers.Conv2D(128,(3,3),activation='relu'))
model.add(keras.layers.MaxPool2D(2,2))

# convolutional layer and maxpool layer 4
model.add(keras.layers.Conv2D(128,(3,3),activation='relu'))
model.add(keras.layers.MaxPool2D(2,2))

# layer to flatten the resulting image array (multiple dimensions) to 1D array
model.add(keras.layers.Flatten())

# hidden layer with 512 neurons and Rectified Linear Unit activation function 
model.add(keras.layers.Dense(512,activation='relu'))

# output layer with single neuron which gives 0 for fish or 1 for non fish
#here we use sigmoid activation function which makes our model output to lie between 0 and 1
model.add(keras.layers.Dense(1,activation='sigmoid'))

In [ ]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
epochs = 10
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=epochs
)

### Analysis of original model

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

# Comparisons

### Model t with good practices and my original model (without proper scaling etc)

In [ ]:
eval_modelt = modelt.evaluate(val_ds)
print(f"Model t (with preprocessing) - Loss: {eval_modelt[0]:.4f}, Accuracy: {eval_modelt[1]:.4f}")

In [ ]:
eval_model = model.evaluate(val_ds)
print(f"Original model - Loss: {eval_model[0]:.4f}, Accuracy: {eval_model[1]:.4f}")

### References 

#### Model

*Sai Balaji | iOS developer,blogger | Aug 28, 2020* "catdog" example

Binary Image classifier CNN using TensorFlow 
- https://medium.com/techiepedia/binary-image-classifier-cnn-using-tensorflow-a3f5d6746697


*Jason Brownlee | May 13, 2019 in Deep Learning for Computer Vision*

How to Develop a CNN From Scratch for CIFAR-10 Photo Classification
- https://machinelearningmastery.com/how-to-develop-a-cnn-from-scratch-for-cifar-10-photo-classification/


*Binh Fahn | ML Engineer | July 6, 2020* "dandeliongrass" example

10 Minutes to Building a CNN Binary Image Classifier in TensorFlow
- https://towardsdatascience.com/10-minutes-to-building-a-cnn-binary-image-classifier-in-tensorflow-4e216b2034aa


#### Data Import

*Aladdin Persson | Intro to Tensorflow on YouTube*

https://www.youtube.com/watch?v=q7ZuZ8ZOErE&list=PLhhyoLH6IjfxVOdVC1P1L5z5azs0XjMsb&index=18


#### General 

*sentdx | CNN and Deep Learning Basics on YouTube*
https://www.youtube.com/watch?v=WvoLTXIjBYU